# Model Independent Review: Chinese Equities

**Main Findings**
1) *Economic Data not lagged properly:*
- trade balance, and economic activit can't simply be adjusted by .shift()
- it's important to get the exact release date (not reference period) to avoid mismat
- PMI data implemented using release date from original excel file

2) *ETF data not suitable for analysis:*
- EWZ price is not total return (e.g., missing dividend payments, ETF borrowing cost, etc...)
- Clear way to demonstrate ability to forecast equity returns is to use liquid IBOV futures (BM&F/B3)

3) ***»»» Critical Look-ahead bias «««*** :*
- Function denoise_data assumes future information is already known by centering moving average
    - When data.rolling(window=window, **center=True**).mean()
    - Missing crutial walk-forward technic to avoid leaking future information into the backtest

4) *Strategy returns are calculated without .shift(1):*
- If model suggests a position of -1, this position should be held for 1-month
- Return of Dec/2024 signal should be calculated with the price of Jan/2025
- Current implementation uses end-of-month information to take a position in the beginning of the month

5) *Hodrick-Prescott (HP) filter:*
- I changed the methodology to make every model OOS, but HP filter suffers from endpoint bias
- It could be useful to explore other models to increase estimators reliability

## Setup

### Imports

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
from functools import reduce
from itertools import product
from scipy.stats import gaussian_kde
from sklearn.decomposition import PCA
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.ar_model import AutoReg

### Constants

In [ ]:
security_id = 'CH1'
date_start = '2008-01-01'
date_end = '2024-12-31'
min_months = 3*12

use_future_data = False
use_future_signal = False

# 资产配置
ASSETS = {
    'CH_EQUITY': 'CSI800',      # 中国股票
    'CH_BOND': '1Oyear_bond',        # 中国债券
    'US_EQUITY': 'PPP500',       # 美国股票
    'GOLD': 'SGE_gold',            # 黄金
    'OIL': 'oil',          # 原油
    'SHORT_BOND': 'short_financing'          # 短期债券
}

### Data

#### Price Data

In [ ]:
price_data = pd.read_csv('China Rolled Return.csv').set_index('date')
price_data.index = pd.to_datetime(price_data.index)
price_data.index.name = 'date'

print("原始数据预览:")
print(price_data.tail())
print(f"\n数据形状: {price_data.shape}")
print(f"列名: {price_data.columns.tolist()}")

# ==================== 计算月度收益 ====================
monthly_returns = {}
for asset_id, col_name in ASSETS.items():
    if col_name in price_data.columns:
        monthly_ret = price_data[col_name].resample('M').last().pct_change(fill_method=None)
        monthly_returns[asset_id] = monthly_ret[date_start:date_end]
    else:
        print(f"警告: 未找到列 '{col_name}'")

monthly_data = pd.DataFrame(monthly_returns)
print("\n月度收益率数据:")
print(monthly_data.tail())


In [ ]:
monthly_data.to_csv('monthly_data.csv')

In [ ]:
price_data = pd.read_csv('China Rolled Return2.csv').set_index('date')
price_data.index = pd.to_datetime(price_data.index)
price_data.index.name = 'date'
price_data.tail(5)
# Create dataframe with monthly total [price + dividends - funding costs (except margin call expenses)] returns
match security_id:
    case 'CH1':
        monthly_data2 = price_data.loc[:,"China"].copy()
        monthly_data2 = monthly_data2.resample('M').last().pct_change(fill_method=None)[date_start:date_end].to_frame('monthly_total_return')

monthly_data2.tail(5)

#### Economic Data

In [ ]:
econ_data = pd.read_csv('china-data.csv')
econ_data.index = econ_data['release_date']
econ_data.index = pd.to_datetime(econ_data.index)
econ_data.index.name = 'date'
econ_data.tail(5)

In [ ]:
econ_data2 = pd.read_csv('data_indicator_review.csv')
econ_data2.index = pd.to_datetime(econ_data2['release_date'])
econ_data2.index.name = 'date'
econ_data2.tail(5)

## Functions

### Data processing

In [ ]:
def adjust_seasonality(data, columns, period=12, model='additive', fill_method='ffill'):
    """
    Removes the seasonal component from specified columns in a time series DataFrame.

    Parameters:
    ----------
    data : pd.DataFrame
        A pandas DataFrame with datetime index and time series columns.
    
    columns : list of str
        List of column names in `data` to apply seasonal adjustment to.
    
    period : int, default=12
        The number of observations per cycle (e.g., 12 for monthly data with yearly seasonality).
    
    model : str, default='additive'
        Type of seasonal decomposition. Must be either 'additive' or 'multiplicative'.
    
    fill_method : str, default='ffill'
        Method used to fill missing values. Options include 'ffill', 'bfill', etc.

    Returns:
    -------
    adjusted_data : pd.DataFrame
        DataFrame of the same shape as the selected columns, with seasonal component removed.
    
    Raises:
    ------
    ValueError:
        If any column in `columns` is not present in the input `data`.

    Notes:
    -----
    - Infinite values are replaced with 0.
    - Missing values are filled using the specified method, and any remaining NaNs are filled with 0.
    - Assumes the input index is datetime-like and regularly spaced.

    Example:
    -------
    >>> adjusted = adjust_seasonality(df, ['sales', 'temperature'], period=12, model='additive')
    """
    adjusted_data = pd.DataFrame(index=data.index)

    for col in columns:
        if col not in data.columns:
            raise ValueError(f"Column '{col}' not found in the input DataFrame.")
        
        # Clean and prepare the series
        series = data[col].copy()
        series.replace([float('inf'), -float('inf')], np.nan, inplace=True)
        series = series.fillna(method=fill_method).dropna()

        # Seasonal decomposition and adjustment
        decomposition = seasonal_decompose(series, model=model, period=period, extrapolate_trend='freq')
        adjusted_data[col] = series - decomposition.seasonal
    
    return adjusted_data

def denoise_data(data, window=3):
    """
    Smooths time series data using a centered moving average to reduce noise.

    Parameters:
    ----------
    data : pd.DataFrame or pd.Series
        Input time series data. Should have a datetime-like index.
    
    window : int, default=3
        Size of the moving average window. Must be an odd integer to ensure centering.
    
    Returns:
    -------
    denoised_data : pd.DataFrame or pd.Series
        Smoothed version of the input data, with NaNs at the edges filled forward and backward.

    Notes:
    -----
    - Uses a centered moving average (`center=True`), which requires sufficient data before and after each point.
    - Edge NaNs from the rolling operation are filled with backward and forward fill.
    - For optimal centering, `window` should be odd; even windows can lead to minor time shifts.

    Example:
    -------
    >>> denoised = denoise_data(df['temperature'], window=5)
    """
    denoised_data = data.rolling(window=window).mean()
    denoised_data = denoised_data.fillna(method='bfill').fillna(method='ffill')
    
    return denoised_data

def standardize_data(data):
    """
    Standardizes numeric features in the DataFrame using z-score normalization.

    Each column is transformed to have mean = 0 and standard deviation = 1.

    Parameters:
    ----------
    data : pd.DataFrame
        A DataFrame of numeric features to standardize. All columns must be numeric.

    Returns:
    -------
    standardized_data : pd.DataFrame
        A DataFrame with the same shape and index as `data`, but with standardized values.

    Notes:
    -----
    - Non-numeric columns should be removed or converted prior to using this function.
    - Missing values (NaNs) in the input will propagate through; they are not handled internally.
    - Uses `StandardScaler` from `sklearn.preprocessing`.

    Example:
    -------
    >>> standardized = standardize_data(df[['sales', 'temperature']])
    """
    scaler = StandardScaler()
    standardized_data = pd.DataFrame(
        scaler.fit_transform(data),
        index=data.index,
        columns=data.columns
    )
    return standardized_data

### Statistical Methods

In [ ]:
def compute_principal_component(data, n_components=1):
    """
    Computes the top principal components of the input dataset using PCA.

    Parameters:
    ----------
    data : pd.DataFrame
        Input DataFrame containing numeric features. Should be standardized beforehand.
    
    n_components : int, default=1
        The number of principal components to compute.

    Returns:
    -------
    principal_df : pd.DataFrame
        A DataFrame containing the computed principal components, with index reset.
        Columns are named 'Principal_Component_1', 'Principal_Component_2', etc.

    Notes:
    -----
    - Input `data` should be standardized (e.g., using `StandardScaler`) before applying PCA.
    - PCA is sensitive to scaling and assumes no missing values.
    - The returned DataFrame has the index reset to default (0, 1, 2, ...).

    Example:
    -------
    >>> pca_df = compute_principal_component(standardized_data, n_components=2)
    """
    pca = PCA(n_components=n_components)
    principal_components = pca.fit_transform(data)
    
    principal_df = pd.DataFrame(
        principal_components,
        index=data.index,
        columns=[f'Principal_Component_{i+1}' for i in range(n_components)]
    )
    
    principal_df.reset_index(inplace=True)
    return principal_df

def calculate_annualized_return(group):
    """
    Calculates the annualized return from a series of daily returns.

    Parameters:
    ----------
    group : pd.DataFrame
        DataFrame containing a 'daily_return' column (e.g., in decimal form, not percent).

    Returns:
    -------
    annualized_return : float
        The annualized return assuming 252 trading days per year.

    Notes:
    -----
    - Assumes daily compounding.
    - Assumes all rows in `group` are consecutive trading days.
    """
    total_return = (1 + group['daily_return']).prod()
    total_months = len(group)
    annualized_return = total_return ** (12/ total_months) - 1
    return annualized_return



def ir(excess_returns: pd.DataFrame, fnorm=12, pct=True, precision=6) -> pd.DataFrame:
    """
    Calculate the annualized mean, standard deviation, and information ratio (IR) 
    for a given DataFrame or Series of returns.

    Parameters:
    excess_returns (pd.DataFrame or pd.Series): Input data containing returns (typically daily).
    fnorm (int): Normalization factor to annualize statistics (default is 252 trading days).

    Returns:
    pd.DataFrame: A DataFrame containing the annualized mean, standard deviation, 
                  and information ratio for each column in `x`.
    """
    # Calculate the mean and standard deviation of the input data
    df_tmp = excess_returns.agg(['mean', 'std'])

    # Annualize the mean by multiplying by the number of periods in a year (e.g., 252 for daily)
    df_tmp.loc['mean'] *= fnorm

    # Annualize the standard deviation by multiplying by the square root of the number of periods
    df_tmp.loc['std'] *= (fnorm ** 0.5)
    
    # Calculate the Information Ratio: annualized mean divided by annualized standard deviation
    df_tmp.loc['ir'] = df_tmp.loc['mean'] / df_tmp.loc['std']

    # Transform mean and std to %
    if pct:
        df_tmp = df_tmp.rename({'mean':r'%_mean', 'std':r'%_std'})
        df_tmp.loc[[r'%_mean',r'%_std']] *= 100
    return df_tmp.round(precision)

def walk_forward_ar_backtest(returns: pd.Series, n_ar: int = 12, initial_window: int = 36) -> pd.Series:
    """
    Perform a walk-forward backtest using an AutoRegressive (AR) model.

    This function simulates a forecasting strategy where at each step in time,
    a model is trained on all data available up to that point (starting from `initial_window`)
    and then used to generate a one-step-ahead forecast. The process continues forward,
    creating a series of forecasted signals.

    Parameters:
    ----------
    returns : pd.Series
        A time series of returns (or other univariate data), indexed by datetime.
    n_ar : int, optional
        Number of lags to use in the AutoRegressive model (default is 12).
    initial_window : int, optional
        Minimum number of initial observations to begin the backtest (default is 36).

    Returns:
    -------
    pd.Series
        A time series of predicted (forecasted) values from the AR model,
        indexed by the time of the forecast.
    """

    forecast_signals = []

    # Ensure the time series is sorted by datetime
    returns = returns.sort_index()

    # Check that the index is a DatetimeIndex
    if not isinstance(returns.index, pd.DatetimeIndex):
        raise ValueError("Index must be a DatetimeIndex.")

    # Walk-forward backtest: iterate through each time step from initial_window onward
    for t in range(initial_window, len(returns) + 1):
        # Use all data up to time t (exclusive) as training data
        train_series = returns.iloc[:t]

        try:
            # Fit an AutoRegressive model with specified number of lags
            model = AutoReg(train_series, lags=n_ar, old_names=False).fit()

            # Generate a one-step-ahead forecast
            predicted = model.predict(start=len(train_series), end=len(train_series))

            # Extract the forecasted signal value
            #signal = predicted.iloc[0]
            signal = predicted[0]
        except Exception:
            # If the model fails to fit or predict, assign NaN and skip to next step
            signal = np.nan
            continue

        # Append the forecasted signal with the corresponding date
        forecast_signals.append([train_series.index[-1], signal])

    # Create a DataFrame of signals and set the date as the index
    df_signal = pd.DataFrame(forecast_signals, columns=['date', 'signal']).set_index('date')

    # Return the signal column as a Series
    return df_signal['signal']

def classify_regime(row, neutral_range=5):
    if abs(row['level']) < neutral_range:
        return 'neutral'
    elif row['level'] >= neutral_range and row['delta'] >= 0:
        return 'expansion'
    elif row['level'] >= neutral_range and row['delta'] < 0:
        return 'normalization'
    elif row['level'] <= -neutral_range and row['delta'] < 0:
        return 'contraction'
    elif row['level'] <= -neutral_range and row['delta'] >= 0:
        return 'recovery'
    else:
        return 'NA'

### Plot Functions

In [ ]:
def plot_factor(data, column, title, xlabel, ylabel):
    """
    Plots a time series column from a DataFrame against a 'date' column.

    Parameters:
    ----------
    data : pd.DataFrame
        A DataFrame containing a 'date' column and the target `column` to plot.
    
    column : str
        Name of the column to plot on the y-axis.
    
    title : str
        Plot title.
    
    xlabel : str
        Label for the x-axis.
    
    ylabel : str
        Label for the y-axis.

    Returns:
    -------
    None
        Displays the plot using matplotlib.

    Notes:
    -----
    - If `data['date']` is a Period type, it is converted to timestamp.
    - The function modifies the original DataFrame by setting 'date' as the index.
    - The plot uses a fixed blue line, with grid and legend enabled.

    Example:
    -------
    >>> plot_factor(df, column='sales', title='Monthly Sales', xlabel='Date', ylabel='Sales')
    """
    # Convert Period to Timestamp if needed
    if isinstance(data['date'].dtype, pd.PeriodDtype):
        data['date'] = data['date'].dt.to_timestamp()

    # Set index to date (modifies original DataFrame)
    data.set_index('date', inplace=True)

    # Plotting
    plt.figure(figsize=(12, 6))
    plt.plot(data.index, data[column], color='blue', label=column)
    plt.title(title, fontsize=14)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.legend()
    plt.grid(True)
    plt.show()

    # Set variable name (modifies original DataFrame)
    data.rename({'Principal_Component_1':ylabel}, axis=1, inplace=True)
    return

def plot_bar(trend_summary, title, xlabel, ylabel, colors=None, figsize=(10, 6)):
    """
    Plots a simple bar chart from a Series or DataFrame summary.

    Parameters:
    ----------
    trend_summary : pd.Series or pd.DataFrame
        Data with categorical index (e.g., labels) and numeric values.
    
    title : str
        Title of the chart.
    
    xlabel : str
        Label for the x-axis.
    
    ylabel : str
        Label for the y-axis.
    
    colors : list of str, optional
        List of colors for the bars. Defaults to ['blue', 'orange'].
    
    figsize : tuple, default=(10, 6)
        Figure size in inches.

    Returns:
    -------
    None
        Displays a matplotlib bar chart.
    """
    categories = trend_summary.index
    values = trend_summary.values

    if colors is None:
        colors = ['blue', 'orange']

    plt.figure(figsize=figsize)
    plt.bar(categories, values, color=colors[:len(categories)])
    plt.title(title, fontsize=14)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.grid(alpha=0.3)
    plt.show()

def plot_box(data, x_col, y_col, title=None, xlabel=None, ylabel=None, palette='Set2', figsize=(10, 6)):
    """
    Draws a box plot comparing distributions of y_col across categories in x_col.

    Parameters:
    ----------
    data : pd.DataFrame
        The dataset containing the x and y columns.
    
    x_col : str
        The column to be used as the category axis (x-axis).
    
    y_col : str
        The column to be used for the value axis (y-axis).
    
    title : str, optional
        Title of the plot.
    
    xlabel : str, optional
        Custom label for the x-axis.
    
    ylabel : str, optional
        Custom label for the y-axis.
    
    palette : str or list, default='Set2'
        Color palette for the boxplot.
    
    figsize : tuple, default=(10, 6)
        Size of the figure in inches.

    Returns:
    -------
    None
        Displays the Seaborn box plot.
    """
    plt.figure(figsize=figsize)
    sns.boxplot(data=data, x=x_col, y=y_col, palette=palette)

    if title:
        plt.title(title, fontsize=14)
    if xlabel:
        plt.xlabel(xlabel, fontsize=12)
    if ylabel:
        plt.ylabel(ylabel, fontsize=12)

    plt.show()

def calculate_annual_statistics(strategy_returns):
    """
    计算分年度策略统计指标
    
    Parameters:
    -----------
    strategy_returns : pd.Series
        月度策略收益率序列（索引为日期）
    
    Returns:
    --------
    pd.DataFrame
        包含各年度统计指标的数据框
    """
    # 确保索引是datetime格式
    if not isinstance(strategy_returns.index, pd.DatetimeIndex):
        strategy_returns.index = pd.to_datetime(strategy_returns.index)
    
    results = []
    
    for year in sorted(strategy_returns.index.year.unique()):
        year_data = strategy_returns[strategy_returns.index.year == year]
        
        # 1. 年化收益率
        annual_return = (1 + year_data).prod() - 1
        
        # 2. 年化波动率（月度波动率 × sqrt(12)）
        annual_vol = year_data.std() * np.sqrt(12)
        
        # 3. 年化夏普比率
        sharpe_ratio = annual_return / annual_vol if annual_vol != 0 else 0
        
        # 4. 最大回撤
        cum_returns = (1 + year_data).cumprod()
        running_max = cum_returns.expanding().max()
        drawdown = (cum_returns - running_max) / running_max
        max_drawdown = drawdown.min()
        
        # 5. 月度胜率
        win_rate = (year_data > 0).sum() / len(year_data)
        
        results.append({
            '年份': year,
            '收益率': annual_return,
            '年化波动率': annual_vol,
            '年化夏普比率': sharpe_ratio,
            '最大回撤': max_drawdown,
            '月度胜率': win_rate
        })
    
    df_annual = pd.DataFrame(results).set_index('年份')
    
    # 格式化输出（可选）
    df_annual_formatted = df_annual.copy()
    df_annual_formatted['收益率'] = df_annual_formatted['收益率'].apply(lambda x: f'{x:.1%}')
    df_annual_formatted['年化波动率'] = df_annual_formatted['年化波动率'].apply(lambda x: f'{x:.1%}')
    df_annual_formatted['年化夏普比率'] = df_annual_formatted['年化夏普比率'].apply(lambda x: f'{x:.2f}')
    df_annual_formatted['最大回撤'] = df_annual_formatted['最大回撤'].apply(lambda x: f'{x:.1%}')
    df_annual_formatted['月度胜率'] = df_annual_formatted['月度胜率'].apply(lambda x: f'{x:.1%}')
    
    return df_annual, df_annual_formatted
    

## Core Model

### Factor Construction

#### Domestic Economy Factor

In [ ]:
# ==================== 国内经济因子 ====================
# 确保 value 列为数值型
econ_data['value'] = pd.to_numeric(econ_data['value'], errors='coerce')

dom_econ_variables = ['china_non-industrial_PMI', 'china_industrial_PMI','loan','property_index']
ylabel = 'Domestic Economic Factor'
nickname = 'dom_econ'

dom_econ = econ_data.query("indicator in @dom_econ_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
# 只保留数值型列
dom_econ = dom_econ.select_dtypes(include=[np.number])
dom_econ = dom_econ.query('date >= @date_start and date <= @date_end').copy()

# --- 【核心修改点】：混合处理数据 ---
# 1. PMI 本身就是景气度指标（荣枯线），保持原始值（Level），不要做差分或增长率，否则会引入巨大噪音。
# 2. Loan（贷款）和 Property_index（房价）是绝对值/累计值，必须转为同比增速（YoY）才能反映“增长动能”。
cols_to_transform = ['loan', 'property_index']
for col in cols_to_transform:
    if col in dom_econ.columns:
        dom_econ[col] = dom_econ[col].pct_change(12)

# 去除因计算同比增速产生的 NaN
dom_econ = dom_econ.dropna()
# ----------------------------------

print("国内经济数据预览 (混合处理后):")
print(dom_econ.tail(5))

# OOS 因子计算
if use_future_data:
    dom_econ_factor = compute_principal_component(
        standardize_data(
            denoise_data(
                adjust_seasonality(dom_econ, dom_econ_variables, period=12), 
                window=12
            )
        ), 
        n_components=1
    )
else:
    dom_econ_factor = []
    for i in range(min_months, dom_econ.shape[0]+1):
        curr_data = dom_econ.iloc[:i].ffill()
        curr_data = curr_data.fillna(curr_data.mean()) 
        df_tmp = compute_principal_component(
            standardize_data(
                denoise_data(
                    adjust_seasonality(
                        dom_econ.iloc[:i].fillna(dom_econ.iloc[:i].mean()), 
                        dom_econ_variables, period=12
                    ), 
                    window=12
                )
            ), 
            n_components=1
        )
        dom_econ_factor.append(df_tmp.iloc[[-1]])
    dom_econ_factor = pd.concat(dom_econ_factor, axis=0)

# 可视化
plot_factor(dom_econ_factor, 'Principal_Component_1', f"{ylabel} Time Series", 'date', ylabel)

# ==================== 生成因子信号 ====================
dom_econ_factor['trend'] = np.where(dom_econ_factor[ylabel].diff()>=0, 'Uptrend', 'Downtrend')
dom_econ_factor[f'{nickname}_signal'] = np.where(
    dom_econ_factor['trend'] == 'Uptrend', 1, 
    np.where(dom_econ_factor['trend'] == 'Downtrend', -1, 0)
)

# 保存因子数据（不合并资产收益率）
dom_econ_factor.to_csv('dom_econ_factor.csv')
print("\n国内经济因子信号:")
print(dom_econ_factor.tail(5))

#### Domestic Currency Factor

In [ ]:
dom_curncy_variables = ['china_1Ybond','china_indicator_reserve-ration','china_market-operations']
ylabel = 'Domestic Currency Factor'
nickname = 'dom_curncy'

dom_curncy = econ_data.query("indicator in @dom_curncy_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
dom_curncy = dom_curncy.query('date >= @date_start and date <= @date_end').copy()
dom_curncy.tail(5)

In [ ]:
# Out-of-sample (OOS) walk-forward methodology
dom_curncy_factor = []
for i in range(min_months, dom_curncy.shape[0]+1):
    df_tmp = -1*(
        denoise_data(
            dom_curncy.iloc[:i].fillna(dom_curncy.iloc[:i].mean()), 
            window=3
        ).diff(12).apply(np.sign)
    )
    dom_curncy_factor.append(df_tmp.iloc[[-1]])
dom_curncy_factor = pd.concat(dom_curncy_factor, axis=0)
dom_curncy_factor['dom_curncy_signal'] = (
    dom_curncy_factor[dom_curncy_variables]
    .mean(axis=1)
    .rolling(window=6, min_periods=3)
    .mean()
)

# Plot OOS Factor
plot_factor(dom_curncy_factor.reset_index(), 'dom_curncy_signal', f"{ylabel} Time Series", 'date', ylabel)

In [ ]:
dom_curncy_factor = pd.concat([dom_curncy_factor, monthly_data2], axis=1).sort_index().dropna()
dom_curncy_factor['trend'] = dom_curncy_factor['dom_curncy_signal'].map({-1: 'Downtrend', 0: 'Neutral', 1:'Uptrend'})
dom_curncy_factor['next_monthly_total_return'] = dom_curncy_factor['monthly_total_return'].shift(-1)
dom_curncy_factor.tail(5)

#### Domestic Credit Factor

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ==================== 国内信用因子 ====================
dom_credit_variables = ['china_medium_long-term_loans']
ylabel = 'Domestic Credit Factor'
nickname = 'dom_credit'

dom_credit = econ_data.query("indicator in @dom_credit_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
dom_credit = dom_credit.query('date >= @date_start and date <= @date_end').copy()
print("原始信用数据列:", dom_credit.columns.tolist())


# 滚动一年新增 = 当前月余额 - 12个月前余额
rolling_sum = dom_credit['china_medium_long-term_loans'].rolling(window=12).sum()
dom_credit['china_medium_long-term_loans_YoY'] = rolling_sum.pct_change(12)

# 去除因 rolling 产生的空值
dom_credit = dom_credit.dropna()
dom_credit_variables = ['china_medium_long-term_loans_YoY']
dom_credit = dom_credit[dom_credit_variables].copy()

print("\n国内信用数据预览:")
print(dom_credit.tail(10))
print("\n信用数据统计:")
print(dom_credit.describe())
# ==================== OOS 因子计算 ====================
dom_credit_factor = []
for i in range(min_months, dom_credit.shape[0]+1):
    curr_data = dom_credit.iloc[:i].ffill().fillna(0)
    df_tmp = compute_principal_component(
        standardize_data(
            denoise_data(
                adjust_seasonality(
                    dom_credit.iloc[:i].fillna(dom_credit.iloc[:i].mean()), 
                    dom_credit_variables, period=12
                ), 
                window=3
            )
        ), 
        n_components=1
    )
    dom_credit_factor.append(df_tmp.iloc[[-1]])

dom_credit_factor = pd.concat(dom_credit_factor, axis=0)

# ==================== 验证因子方向 ====================
print("\n因子方向验证:")
print("正值表示信用扩张（贷款增加），应利好股票")
print("负值表示信用收缩（贷款减少），应利好债券")
print(dom_credit_factor.tail(10))

# 可视化因子
plot_factor(dom_credit_factor, 'Principal_Component_1', f"{ylabel} Time Series", 'date', ylabel)

# ==================== 生成因子信号 ====================
dom_credit_factor['trend'] = np.where(dom_credit_factor[ylabel].diff()>=0, 'Uptrend', 'Downtrend')
dom_credit_factor[f'{nickname}_signal'] = np.where(
    dom_credit_factor['trend'] == 'Uptrend', 1, 
    np.where(dom_credit_factor['trend'] == 'Downtrend', -1, 0)
)

# 保存因子
dom_credit_factor.to_csv('dom_credit_factor.csv')
print("\n国内信用因子信号:")
print(dom_credit_factor[[ylabel, 'trend', f'{nickname}_signal']].tail(5))

# ==================== 因子有效性验证（可选）====================
print("\n" + "="*80)
print("因子有效性分析")
print("="*80)

factor_effectiveness = {}
for asset_id, col_name in ASSETS.items():
    if asset_id in monthly_data.columns:
        # 合并因子信号和资产收益
        df_test = pd.concat([
            dom_credit_factor[f'{nickname}_signal'],
            monthly_data[asset_id]
        ], axis=1).dropna()
        
        # 计算下期收益
        df_test['next_return'] = df_test[asset_id].shift(-1)
        df_test = df_test.dropna()
        
        if len(df_test) > 0:
            # 统计分析
            correlation = df_test[f'{nickname}_signal'].corr(df_test['next_return'])
            signal_return = (df_test[f'{nickname}_signal'] * df_test['next_return']).mean()
            win_rate = (df_test[f'{nickname}_signal'] * df_test['next_return'] > 0).mean()
            
            factor_effectiveness[asset_id] = {
                'correlation': correlation,
                'signal_return': signal_return,
                'win_rate': win_rate,
                'asset_name': col_name
            }
            
            print(f"{col_name:15s}: 相关={correlation:6.3f}, 信号收益={signal_return*100:6.2f}%, 胜率={win_rate*100:5.1f}%")

# ==================== 可视化（使用基准资产）====================
# 选择中证800作为基准进行可视化
benchmark_asset = 'CH_EQUITY'

if benchmark_asset in monthly_data.columns:
    # 合并因子和基准收益
    df_viz = pd.concat([
        dom_credit_factor[[f'{nickname}_signal', 'trend']],
        monthly_data[[benchmark_asset]]
    ], axis=1).dropna()
    
    df_viz['next_return'] = df_viz[benchmark_asset].shift(-1)
    df_viz = df_viz.dropna()
    
    # 可视化
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    
    # KDE图
    sns.kdeplot(data=df_viz, x='next_return', hue='trend', fill=True, ax=ax[0])
    ax[0].set_title(f'中证800下期收益分布 by 信用因子趋势')
    ax[0].set_xlabel('下期月收益率')
    ax[0].axvline(x=0, color='red', linestyle='--', alpha=0.5)
    
    # 累计收益图
    cumulative_return = (df_viz[f'{nickname}_signal'] * df_viz['next_return']).fillna(0).add(1).cumprod()
    cumulative_return.plot(ax=ax[1], linewidth=2, color='green')
    ax[1].set_title('信用因子策略累计收益（中证800）')
    ax[1].set_ylabel('累计收益倍数')
    ax[1].grid(True, alpha=0.3)
    ax[1].axhline(y=1, color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig('dom_credit_factor_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 打印策略统计
    if len(cumulative_return) > 0:
        total_return = (cumulative_return.iloc[-1] - 1) * 100
        annualized_return = (cumulative_return.iloc[-1] ** (12/len(cumulative_return)) - 1) * 100
        win_rate = (df_viz[f'{nickname}_signal'] * df_viz['next_return'] > 0).mean() * 100
        
        print(f"\n信用因子策略统计（基于中证800）:")
        print(f"  总收益: {total_return:.2f}%")
        print(f"  年化收益: {annualized_return:.2f}%")
        print(f"  胜率: {win_rate:.2f}%")
        print(f"  观测数: {len(df_viz)}")

print(f"\n✓ 因子已保存至: dom_credit_factor.csv")

#### Domestic Inflation Factor

In [ ]:
dom_inflation_variables = ['china_inflation_PMI']
ylabel = 'Domestic Inflation Factor'
nickname = 'dom_inflation'

dom_inflation = econ_data.query("indicator in @dom_inflation_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
dom_inflation = dom_inflation.query('date >= @date_start and date <= @date_end').copy()

dom_inflation_variables = ['china_inflation_PMI']
dom_inflation = dom_inflation[dom_inflation_variables].copy()
dom_inflation.tail(5)

In [ ]:
# Out-of-sample (OOS) walk-forward methodology
dom_inflation_factor = []
for i in range(min_months, dom_inflation.shape[0]+1):
    curr_data = dom_inflation.iloc[:i].ffill().fillna(0)
    df_tmp = compute_principal_component(
        denoise_data(
            adjust_seasonality(
                dom_inflation.iloc[:i].fillna(dom_inflation.iloc[:i].mean()),
                dom_inflation_variables, period=12
                )
            ,
            window=3
        ),
        n_components=1
    )
    dom_inflation_factor.append(df_tmp.iloc[[-1]])
dom_inflation_factor = pd.concat(dom_inflation_factor, axis=0)

# Plot OOS Factor
plot_factor(dom_inflation_factor, 'Principal_Component_1', f"{ylabel} Time Series", 'date', ylabel)

In [ ]:
# 1. 计算环比变化 (Change)
factor_delta = dom_inflation_factor[ylabel].diff()

# 2. 计算滚动3年(36个月)的标准差 (Benchmark)
# 使用 shift(1) 保证使用过去36个月的数据标准来衡量当月，避免未来函数
rolling_std = factor_delta.rolling(window=36).std().shift(1)

# 3. 状态分类
dom_inflation_factor['trend'] = 'Neutral'

# 上行：环比变化 > 1倍标准差
# fillna(False) 是为了处理序列开始时的 NaN
dom_inflation_factor.loc[(factor_delta > rolling_std).fillna(False), 'trend'] = 'Uptrend'

# 下行：环比变化 < -1倍标准差
dom_inflation_factor.loc[(factor_delta < -rolling_std).fillna(False), 'trend'] = 'Downtrend'

# Build signal
dom_inflation_factor['dom_inflation_signal'] = dom_inflation_factor['trend'].map({'Downtrend':-1, 'Neutral': 0, 'Uptrend':+1})
dom_inflation_factor['dom_inflation_signal'] = dom_inflation_factor['dom_inflation_signal'].astype(int)
# Combine with returns
dom_inflation_factor = pd.concat([dom_inflation_factor, monthly_data2], axis=1).sort_index().dropna()
dom_inflation_factor['next_monthly_total_return'] = dom_inflation_factor['monthly_total_return'].shift(-1)
dom_inflation_factor.tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=dom_inflation_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((dom_inflation_factor['dom_inflation_signal'] * dom_inflation_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### Global Economy Factor

In [ ]:
global_econ_variables = ['usa_pmi','china_pmi_new_export_orders','korea_exports','copper_lme','gold']
ylabel = 'Global Economic Factor'
nickname = 'global_econ'

global_econ = econ_data2.query("indicator in @global_econ_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
global_econ = global_econ.query('date >= @date_start and date <= @date_end').copy()
# Build global_econ metrics
global_econ[['usa_pmi','china_pmi_new_export_orders']] -= 50
global_econ['korea_exports'] = global_econ['korea_exports'].pct_change(12).rolling(6).mean()
global_econ['copper/gold'] = (global_econ['copper_lme'] / global_econ['gold']).apply(np.log).rolling(6).mean()

global_econ_variables = ['usa_pmi','china_pmi_new_export_orders','korea_exports','copper/gold']
global_econ = global_econ[global_econ_variables].copy()
print(global_econ.tail(5))

In [ ]:
# Out-of-sample (OOS) walk-forward methodology
global_econ_factor = []
for i in range(min_months, global_econ.shape[0]+1):
    curr_data = global_econ.iloc[:i].ffill().fillna(0)
    df_tmp = compute_principal_component(
        standardize_data(
            denoise_data(
                adjust_seasonality(
                    global_econ.iloc[:i].fillna(global_econ.iloc[:i].mean()),
                    global_econ_variables, period=12
                    ),
                window=12
            ),
        ),
        n_components=1
    )
    global_econ_factor.append(df_tmp.iloc[[-1]])
global_econ_factor = pd.concat(global_econ_factor, axis=0)

# Plot OOS Factor
plot_factor(global_econ_factor, 'Principal_Component_1', f"{ylabel} Time Series", 'date', ylabel)

In [ ]:
global_econ_factor.columns = [ylabel]

# [关键修改]：Z-Score 标准化后乘以 0.5
# 标准正态分布 99% 在 +/- 3 之间。乘以 0.5 后，范围变为 +/- 1.5。
# 结合数据的自然偏离，这通常能完美落在 -2 到 1 的区间内。
factor_mean = global_econ_factor[ylabel].mean()
factor_std = global_econ_factor[ylabel].std()
global_econ_factor[ylabel] = ((global_econ_factor[ylabel] - factor_mean) / factor_std) * 0.5

# 6. 绘制因子图 (重置索引以避免绘图报错)
plot_df = global_econ_factor.reset_index()
plot_factor(plot_df, ylabel, f"{ylabel} Time Series (Smoothed & Scaled)", 'date', ylabel)

# 7. 构建信号与合并收益 (修复报错的核心部分)
# 确保索引格式一致，避免 concat 结果为空
global_econ_factor.index = pd.to_datetime(global_econ_factor.index)
monthly_data2.index = pd.to_datetime(monthly_data2.index)

# 计算差分 (Trend)
# 注意：直接用 .diff() 可能会产生 DataFrame，指定列名更安全
global_econ_factor['diff'] = global_econ_factor[ylabel].diff()

# 定义趋势
global_econ_factor['trend'] = 'Neutral'
global_econ_factor.loc[global_econ_factor['diff'] > 0, 'trend'] = 'Uptrend'
global_econ_factor.loc[global_econ_factor['diff'] < 0, 'trend'] = 'Downtrend'

# 生成信号数值
global_econ_factor['global_econ_signal'] = global_econ_factor['trend'].map({'Downtrend':-1, 'Neutral':0, 'Uptrend':1})

# 合并收益数据 (使用 inner join 确保日期匹配)
combined_df = pd.concat([global_econ_factor, monthly_data2], axis=1, join='inner').sort_index()

# 计算下期收益 (Next Return)
combined_df['next_monthly_total_return'] = combined_df['monthly_total_return'].shift(-1)

# [关键]：在绘图前删除包含 NaN 的行 (主要是最后一行因为 shift(-1) 变成了 NaN)
combined_df = combined_df.dropna()

print("Combined Data Tai[LOCAL_PATH_REMOVED]", combined_df[['trend', 'next_monthly_total_return']].tail())

# 8. 绘图分析
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# KDE Plot
# 此时 combined_df 已经 dropna，且包含 next_monthly_total_return 列，不会报错
sns.kdeplot(data=combined_df, x='next_monthly_total_return', hue='trend', fill=True, common_norm=False, ax=ax[0])
ax[0].set_title('KDE: Next Monthly Return by Trend')
ax[0].grid(True, alpha=0.3)

# Cumulative Return Plot
# 策略收益：信号 * 下期收益
strategy_ret = combined_df['global_econ_signal'] * combined_df['next_monthly_total_return']
(strategy_ret.fillna(0).add(1).cumprod() - 1).plot(title='Strategy Cumulative Return', ax=ax[1])
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Global Currency Factor

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 参数设置
global_curncy_variables = ['treasury_yld_1y', 'fed_securities_bs']
ylabel = 'Global Currency Factor'
nickname = 'global_curncy'

# 2. 数据提取
global_curncy = econ_data2.query("indicator in @global_curncy_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
global_curncy = global_curncy.query('date >= @date_start and date <= @date_end').copy()

# Build global_curncy variables
global_curncy['smoothed_yield'] = global_curncy['treasury_yld_1y'].rolling(window=3).mean()
global_curncy['rate_change'] = (global_curncy['smoothed_yield'] - global_curncy['smoothed_yield'].shift(1)).fillna(0.)

# 【核心修正】：利率上升代表紧缩，信号应为 -1，所以使用 -np.sign
global_curncy['yield_signal'] = (global_curncy['rate_change']
                                     .apply(lambda x: np.nan if abs(x) < 0.03 else -np.sign(x)) 
                                     .ffill().fillna(0.).astype(int)
                                )

global_curncy['fed_securities_bs'] /= 1e3 # now in Billions
global_curncy['holdings_signal'] = 2*(global_curncy['fed_securities_bs'].diff()>30)-1


# 筛选列
global_curncy_variables = ['smoothed_yield','yield_signal','holdings_signal']
global_curncy = global_curncy[global_curncy_variables].copy()

# 4. 创建综合信号 (Factor Construction)
# 逻辑：如果利率极低 (<0.5)，主要看央行扩表(holdings_signal)；否则主要看利率趋势(yield_signal)
global_curncy['global_curncy_signal'] = global_curncy.apply(
                                            lambda row: row['holdings_signal'] 
                                            if row['smoothed_yield'] < 0.5
                                            else row['yield_signal'], axis=1
                                        )

# 映射趋势标签
global_curncy['trend'] = global_curncy['global_curncy_signal'].map({-1:'Downtrend', 0:'Neutral', 1:'Uptrend'})

# 5. 合并收益数据
# 注意：先合并，再计算下期收益，最后 dropna，防止数据错位
global_curncy_factor = pd.concat([global_curncy, monthly_data2], axis=1, join='inner').sort_index()
global_curncy_factor['next_monthly_total_return'] = global_curncy_factor['monthly_total_return'].shift(-1)
global_curncy_factor = global_curncy_factor.dropna()

print(global_curncy_factor[['smoothed_yield', 'global_curncy_signal', 'trend']].tail(5))

# ==========================================
# 新增：绘制因子趋势图 (Factor Trend Plot)
# ==========================================
fig, ax1 = plt.subplots(figsize=(12, 5))

# 绘制底层核心变量：平滑后的收益率 (右轴)
ax2 = ax1.twinx()
ax2.plot(global_curncy_factor.index, global_curncy_factor['smoothed_yield'], 
         color='gray', alpha=0.3, linestyle='--', label='Smoothed 1Y Yield (Right)')
ax2.set_ylabel('1Y Treasury Yield (%)', color='gray')

# 绘制因子信号 (左轴，阶梯图)
# 使用 step-post 确保信号变化时刻对齐
ax1.step(global_curncy_factor.index, global_curncy_factor['global_curncy_signal'], 
         where='post', color='blue', linewidth=2, label='Currency Signal (Left)')

# 填充颜色以区分趋势
ax1.fill_between(global_curncy_factor.index, global_curncy_factor['global_curncy_signal'], 0, 
                 where=(global_curncy_factor['global_curncy_signal'] > 0), 
                 color='green', alpha=0.1, step='post', label='Uptrend Zone')
ax1.fill_between(global_curncy_factor.index, global_curncy_factor['global_curncy_signal'], 0, 
                 where=(global_curncy_factor['global_curncy_signal'] < 0), 
                 color='red', alpha=0.1, step='post', label='Downtrend Zone')

ax1.set_title(f'{ylabel} Time Series & Regime Switch', fontsize=12)
ax1.set_ylabel('Signal (-1: Down, 0: Neutral, 1: Up)')
ax1.set_yticks([-1, 0, 1])
ax1.set_yticklabels(['Downtrend', 'Neutral', 'Uptrend'])
ax1.grid(True, axis='y', linestyle=':', alpha=0.6)

# 合并图例
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc='upper left')

plt.show()

# ==========================================
# 原有的评估图 (KDE & Cumulative Return)
# ==========================================
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# KDE Plot
sns.kdeplot(data=global_curncy_factor, x='next_monthly_total_return', hue='trend', 
            fill=True, palette={'Uptrend':'green', 'Downtrend':'red', 'Neutral':'gray'}, 
            warn_singular=False, ax=ax[0])
ax[0].set_title('KDE: Next Monthly Return by Trend')
ax[0].grid(True, alpha=0.3)

# Cumulative Return Plot
# 策略逻辑：信号 * 下期收益
strategy_ret = (global_curncy_factor['global_curncy_signal'] * global_curncy_factor['next_monthly_total_return'])
strategy_cum = strategy_ret.shift(1).fillna(0.).add(1).cumprod()

strategy_cum.plot(title='Strategy Cumulative Return (Signal * Next Return)', ax=ax[1])
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Global Inflation Factor

In [ ]:
global_inflation_variables = ['us_10y_breakeven_inflation']
ylabel = 'Global Inflation Factor'
nickname = 'global_inflation'
global_inflation = econ_data2.query("indicator in @global_inflation_variables").pivot_table('value','date','indicator').sort_index()
global_inflation = global_inflation.query('date >= @date_start and date <= @date_end').sort_index().copy()

# Smooth data with HP filter (OOS)
df_tmp = []
idx = global_inflation.resample('M').last().index
for i, dt in enumerate(idx[12:]):
    # 这里保持不变，获取趋势项
    trend = (hpfilter(global_inflation[['us_10y_breakeven_inflation']]
                      .query("date <= @dt"), lamb=129_600)[1].iloc[-1]
            )
    df_tmp.append((dt, trend))

global_inflation = (pd.DataFrame(df_tmp, columns=['date','smooth_us10yBE'])
                        .set_index('date').resample('M').last())

# 【改动 1】：将因子(z)的计算从后面移到绘图之前
# 解释：右图的蓝线是围绕0波动的因子，而不是原始的通胀率。
# 原代码的逻辑是 (值 - 均值) / 标准差，这正是我们需要画的因子。
global_inflation['global_inflation_factor'] = (
    (global_inflation['smooth_us10yBE'] - global_inflation['smooth_us10yBE'].ewm(halflife=12,min_periods=12).mean()) /
    global_inflation['smooth_us10yBE'].ewm(halflife=12,min_periods=12).std()
    ) 


# 【改动 2】：修改绘图对象
# 原代码：plot_factor(..., 'smooth_us10yBE', ...) -> 画的是趋势（左图）
# 新代码：plot_factor(..., 'global_inflation_factor', ...) -> 画的是因子（右图）
plot_factor(global_inflation.reset_index(), 'global_inflation_factor', f"{ylabel} Time Series", 'date', ylabel)

# Build trend
# 【改动 3】：后续逻辑引用新的列名 'global_inflation_factor' (即原来的 'z')
global_inflation['trend'] = 'Neutral'
idx = global_inflation['global_inflation_factor'] > 1
global_inflation.loc[idx, 'trend'] = 'Uptrend'
idx = global_inflation['global_inflation_factor'] < -1
global_inflation.loc[idx, 'trend'] = 'Downtrend'

# Build signal
global_inflation['global_inflation_signal'] = global_inflation['trend'].map({'Downtrend':-1, 'Neutral': 0, 'Uptrend':+1})
global_inflation['global_inflation_signal'] = global_inflation['global_inflation_signal'].astype(int)

# Combine with returns
global_inflation_factor = pd.concat([global_inflation, monthly_data2], axis=1).sort_index().dropna()
global_inflation_factor['next_monthly_total_return'] = global_inflation_factor['monthly_total_return'].shift(-1)
global_inflation_factor.tail(5)

# (绘图部分代码保持不变)
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=global_inflation_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((global_inflation_factor['global_inflation_signal'] * global_inflation_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### US Dollar Cycle

In [ ]:
dollar_variables = ['dollar_index_dxy']
ylabel = 'Dollar Factor'
nickname = 'dollar_cycle'
dollar_cycle = econ_data2.query("indicator in @dollar_variables").pivot_table('value','date','indicator').sort_index()
dollar_cycle = dollar_cycle.query('date >= @date_start and date <= @date_end').sort_index().copy()
# Smooth data with HP filter (OOS)
df_tmp = []
idx = dollar_cycle.resample('M').last().index
for i, dt in enumerate(idx[12:]):
    trend = (hpfilter(dollar_cycle[['dollar_index_dxy']]
                      .query("date <= @dt"), lamb=129_600)[1].iloc[-1]
            )
    df_tmp.append((dt, trend))
dollar_cycle = (pd.DataFrame(df_tmp, columns=['date','dollar_index_dxy'])
                .set_index('date').resample('M').last())
# Plot OOS Breakeven Inflation
plot_factor(dollar_cycle.reset_index(), 'dollar_index_dxy', f"{ylabel} Time Series", 'date', ylabel)

In [ ]:
# Build ranges
dollar_cycle['z'] = (
    (dollar_cycle['dollar_index_dxy'] - dollar_cycle['dollar_index_dxy'].ewm(halflife=12,min_periods=12).mean()) /
    dollar_cycle['dollar_index_dxy'].ewm(halflife=12,min_periods=12).std()
    ) * (12**0.5)
# Build trend
dollar_cycle['trend'] = 'Neutral'
idx = dollar_cycle['z'] > 1
dollar_cycle.loc[idx, 'trend'] = 'Uptrend'
idx = dollar_cycle['z'] < -1
dollar_cycle.loc[idx, 'trend'] = 'Downtrend'
# Build signal
dollar_cycle['dollar_cycle_signal'] = dollar_cycle['trend'].map({'Downtrend':-1, 'Neutral': 0, 'Uptrend':+1})
dollar_cycle['dollar_cycle_signal'] = dollar_cycle['dollar_cycle_signal'].astype(int)
# Combine with returns
dollar_cycle_factor = pd.concat([dollar_cycle, monthly_data2], axis=1).sort_index().dropna()
dollar_cycle_factor['next_monthly_total_return'] = dollar_cycle_factor['monthly_total_return'].shift(-1)
dollar_cycle_factor.tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=dollar_cycle_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((dollar_cycle_factor['dollar_cycle_signal'] * dollar_cycle_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### Global Financial Risk

In [ ]:
global_fin_risk_variables = ['ofr_fsi']
ylabel = 'Global Financial Risk Factor'
nickname = 'global_fin_risk'

global_fin_risk = econ_data2.query("indicator in @global_fin_risk_variables").pivot_table('value','date','indicator').sort_index().resample('M').last()
global_fin_risk = global_fin_risk.query('date >= @date_start and date <= @date_end').copy()
global_fin_risk.tail(5)

# Build ranges
# 【修改】：删除了末尾的 * (12**0.5)，防止因子波动幅度过大，使其保持在标准正态分布区间
global_fin_risk['z'] = (
    (global_fin_risk['ofr_fsi'] - global_fin_risk['ofr_fsi'].ewm(halflife=36,min_periods=12).mean()) /
    global_fin_risk['ofr_fsi'].ewm(halflife=36,min_periods=12).std()
    ) 

# 【新增】：在此处绘制因子趋势图
plot_factor(global_fin_risk.reset_index(), 'z', f"{ylabel} Time Series", 'date', ylabel)

# Build trend
global_fin_risk['trend'] = 'Neutral'
idx = global_fin_risk['z'] > 1
global_fin_risk.loc[idx, 'trend'] = 'Uptrend'
idx = global_fin_risk['z'] < -1
global_fin_risk.loc[idx, 'trend'] = 'Downtrend'

# Build signal
global_fin_risk['fin_risk_signal'] = global_fin_risk['trend'].map({'Downtrend':-1, 'Neutral': 0, 'Uptrend':+1})
global_fin_risk['fin_risk_signal'] = global_fin_risk['fin_risk_signal'].astype(int)

# Combine with returns
global_fin_risk_factor = pd.concat([global_fin_risk, monthly_data2], axis=1).sort_index().dropna()
global_fin_risk_factor['next_monthly_total_return'] = global_fin_risk_factor['monthly_total_return'].shift(-1)
global_fin_risk_factor.tail(5)

fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=global_fin_risk_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((global_fin_risk_factor['fin_risk_signal'] * global_fin_risk_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

#### Autocorrelation

In [ ]:
monthly_data2['monthly_total_return']

In [ ]:
ylabel = 'Auto Correlation Factor'
ar_factor = walk_forward_ar_backtest(monthly_data2['monthly_total_return'].iloc[1:], n_ar=12, initial_window=36).to_frame('ar_factor')
# Plot OOS Factor
plot_factor(ar_factor.apply(np.sign).reset_index(), 'ar_factor', f"{ylabel} Time Series", 'date', ylabel)
ar_factor

In [ ]:
ar_factor['ar_signal'] = ar_factor['ar_factor'].apply(np.sign)
ar_factor['trend'] = ar_factor['ar_signal'].map({-1:'Downtrend', 0:'Neutral', 1:'Uptrend'})
# Combine with returns
ar_factor = pd.concat([ar_factor, monthly_data2], axis=1).sort_index().dropna()
ar_factor['next_monthly_total_return'] = ar_factor['monthly_total_return'].shift(-1)
ar_factor.tail(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))
_ = sns.kdeplot(data=ar_factor, x='next_monthly_total_return', hue='trend', fill=True, ax=ax[0])
_ = ax[0].set_title('KDE Plot of next_monthly_total_return by Trend')
_ = ((ar_factor['ar_signal'] * ar_factor['next_monthly_total_return'])
         .shift(1).fillna(0.).add(1).cumprod().plot(title='next_monthly_total_return: Cumulative Return', ax=ax[1])
    )
plt.tight_layout()

### Backtest

In [ ]:
FACTOR_WEIGHTS = {
    'CH_EQUITY': {  # 中证800 - 权益类
        'dom_econ_signal': 1,           # 国内景气：利好
        'dom_curncy_signal': 1,         # 国内货币：利好
        'dom_credit_signal': 1,         # 国内信用：利好
        'dom_inflation_signal': -1,     # 国内通胀：利空
        'global_econ_signal': 0,        # 全球景气：无方向
        'global_curncy_signal': 1,      # 全球货币：利好
        'global_inflation_signal': 0,   # 全球通胀：无方向
        'dollar_cycle_signal': -1,      # 美元周期：利空
        'fin_risk_signal': -1,          # 全球金融风险：利空
        'ar_signal': 0,                 # 表中无此项
    },

    'CH_BOND': {  # 10年国债 - 与股票负相关策略
        'dom_econ_signal': -1,          # 国内景气：利空
        'dom_curncy_signal': 1,         # 国内货币：利好
        'dom_credit_signal': -1,        # 国内信用：利空
        'dom_inflation_signal': -1,     # 国内通胀：利空
        'global_econ_signal': 0,        # 全球景气：无方向
        'global_curncy_signal': 0,      # 全球货币：无方向
        'global_inflation_signal': 0,   # 全球通胀：无方向
        'dollar_cycle_signal': 1,       # 美元周期：利好
        'fin_risk_signal': 1,           # 全球金融风险：利好（避险）
        'ar_signal': 0,                 # 表中无此项
    },

    'US_EQUITY': {  # 标普500 - 全球宏观主导
        'dom_econ_signal': 0,           # 国内景气：无方向
        'dom_curncy_signal': 0,         # 国内货币：无方向
        'dom_credit_signal': 0,         # 国内信用：无方向
        'dom_inflation_signal': 0,      # 国内通胀：无方向
        'global_econ_signal': 1,        # 全球景气：利好
        'global_curncy_signal': 1,      # 全球货币：利好
        'global_inflation_signal': 0,   # 全球通胀：无方向
        'dollar_cycle_signal': 0,       # 美元周期：无方向
        'fin_risk_signal': -1,          # 全球金融风险：利空
        'ar_signal': 0,                 # 表中无此项
    },

    'GOLD': {  # 黄金 - 避险+通胀对冲
        'dom_econ_signal': 0,           # 国内景气：无方向
        'dom_curncy_signal': 0,         # 国内货币：无方向
        'dom_credit_signal': 0,         # 国内信用：无方向
        'dom_inflation_signal': 0,      # 国内通胀：无方向
        'global_econ_signal': -1,       # 全球景气：利空（避险资产）
        'global_curncy_signal': 0,      # 全球货币：无方向
        'global_inflation_signal': 0,   # 全球通胀：无方向
        'dollar_cycle_signal': 0,       # 美元周期：无方向
        'fin_risk_signal': 1,           # 全球金融风险：利好（避险）
        'ar_signal': 0,                 # 表中无此项
    },

    'OIL': {  # 原油 - 全球经济周期
        'dom_econ_signal': 0,           # 国内景气：无方向
        'dom_curncy_signal': 0,         # 国内货币：无方向
        'dom_credit_signal': 0,         # 国内信用：无方向
        'dom_inflation_signal': 0,      # 国内通胀：无方向
        'global_econ_signal': 1,        # 全球景气：利好
        'global_curncy_signal': 1,      # 全球货币：利好
        'global_inflation_signal': 1,   # 全球通胀：利好
        'dollar_cycle_signal': -1,      # 美元周期：利空
        'fin_risk_signal': -1,          # 全球金融风险：利空
        'ar_signal': 0,                 # 表中无此项
    },

    'SHORT_BOND': {  # 短融 - 现金等价物
        'dom_econ_signal': 0,           # 国内景气：无方向
        'dom_curncy_signal': -1,        # 国内货币：利空
        'dom_credit_signal': 0,         # 国内信用：无方向
        'dom_inflation_signal': 1,      # 国内通胀：利好
        'global_econ_signal': 0,        # 全球景气：无方向
        'global_curncy_signal': -1,     # 全球货币：利空
        'global_inflation_signal': -1,  # 全球通胀：利空
        'dollar_cycle_signal': 0,       # 美元周期：无方向
        'fin_risk_signal': 0,           # 全球金融风险：无方向
        'ar_signal': 0,                 # 表中无此项
    },
}

In [ ]:
# ==================== 1. 准备因子信号数据 ====================
df_signal = pd.concat([
    dom_econ_factor['dom_econ_signal'], 
    dom_curncy_factor['dom_curncy_signal'], 
    dom_credit_factor['dom_credit_signal'], 
    dom_expectation['dom_expectation_signal'],
    dom_inflation_factor['dom_inflation_signal'],
    dom_fx_factor['dom_fx_signal'],
    global_econ_factor['global_econ_signal'],
    global_curncy_factor['global_curncy_signal'],
    global_inflation['global_inflation_signal'],
    dollar_cycle_factor['dollar_cycle_signal'],
    global_fin_risk_factor['fin_risk_signal'],
    ar_factor['ar_signal']
], axis=1)

df_signal = df_signal.loc[df_signal.count(axis=1) >= df_signal.shape[1]//2].copy()
print(f"因子信号数据形状: {df_signal.shape}\n")

# ==================== 2. 计算各资产策略 ====================
print("="*80)
print("各资产策略绩效")
print("="*80)

asset_strategies = {}
asset_positions = {}

for asset_id in ASSETS.keys():
    if asset_id not in monthly_data.columns:
        continue
    
    # 该资产的因子权重
    df_weight = FACTOR_WEIGHTS[asset_id]
    
    # 计算仓位和策略收益
    weights = pd.Series(df_weight)
    weights = weights / weights.abs().sum()
    df_pos = df_signal.mul(weights, axis=1).sum(axis=1)
    df_strat = df_pos.shift(1) * monthly_data[asset_id]
    
    # 保存结果
    asset_positions[asset_id] = df_pos
    asset_strategies[asset_id] = df_strat
    
    # 打印绩效 
    sharpe = ir(df_strat, fnorm=12, precision=2)
    # 从 Series 中提取数值
    if isinstance(sharpe, pd.Series):
        sharpe_value = sharpe.iloc[0]
    else:
        sharpe_value = sharpe
    print(f"{ASSETS[asset_id]:15s} ({asset_id:12s}): Sharpe = {sharpe_value:.2f}")

df_positions = pd.DataFrame(asset_positions)
df_strategies = pd.DataFrame(asset_strategies)

# ==================== 3. 计算组合策略 ====================
PORTFOLIO_WEIGHTS = {
    'CH_EQUITY': 0.10,
    'CH_BOND': 0.40,
    'US_EQUITY': 0.05,
    'GOLD': 0.30,
    'OIL': 0.05,
    'CREDIT': 0.10,
}

portfolio_weights = pd.Series(PORTFOLIO_WEIGHTS)
df_portfolio = (df_strategies * portfolio_weights).sum(axis=1)

print("\n" + "="*80)
print("组合策略绩效")
print("="*80)

# 打印组合 Sharpe Ratio
portfolio_sharpe = ir(df_portfolio, fnorm=12, precision=2)
if isinstance(portfolio_sharpe, pd.Series):
    portfolio_sharpe_value = portfolio_sharpe.iloc[0]
else:
    portfolio_sharpe_value = portfolio_sharpe
print(f"组合 Sharpe Ratio: {portfolio_sharpe_value:.2f}")

# ==================== 4. 分年度统计 ====================
df_annual_stats, df_annual_stats_formatted = calculate_annual_statistics(df_portfolio)

print("\n组合策略分年度绩效统计")
print("=" * 80)
print(df_annual_stats_formatted)

# ==================== 5. 可视化 ====================
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 图1：各资产累计收益
ax1 = axes[0]
for asset_id in ASSETS.keys():
    if asset_id in df_strategies.columns:
        cumret = df_strategies[asset_id].add(1).cumprod()
        sharpe = ir(df_strategies[asset_id], fnorm=12, precision=2)
        sharpe_value = sharpe.iloc[0] if isinstance(sharpe, pd.Series) else sharpe
        ax1.plot(cumret.index, cumret.values, 
                label=f"{ASSETS[asset_id]} (SR={sharpe_value:.2f})", linewidth=1.5)
ax1.set_title('各资产策略累计收益', fontsize=14, fontweight='bold')
ax1.legend(loc='best', fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_ylabel('累计收益')

# 图2：组合累计收益
ax2 = axes[1]
portfolio_cumret = df_portfolio.add(1).cumprod()
ax2.plot(portfolio_cumret.index, portfolio_cumret.values, 
         linewidth=2.5, color='darkblue', label=f'组合策略 (SR={portfolio_sharpe_value:.2f})')
ax2.fill_between(portfolio_cumret.index, 1, portfolio_cumret.values, alpha=0.3)
ax2.set_title('组合策略累计收益', fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(alpha=0.3)
ax2.set_ylabel('累计收益')

# 图3：分年度收益柱状图
ax3 = axes[2]
annual_returns = df_annual_stats['收益率']
colors = ['green' if x > 0 else 'red' for x in annual_returns]
ax3.bar(annual_returns.index, annual_returns.values, color=colors, alpha=0.7, edgecolor='black')
ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax3.set_title('组合策略分年度收益率', fontsize=14, fontweight='bold')
ax3.set_xlabel('年份')
ax3.set_ylabel('年度收益率')
ax3.grid(alpha=0.3, axis='y')
# 在柱状图上标注数值
for year, ret in annual_returns.items():
    ax3.text(year, ret, f'{ret:.1%}', ha='center', 
             va='bottom' if ret > 0 else 'top', fontsize=8)

plt.tight_layout()
plt.savefig('multi_asset_performance.png', dpi=300, bbox_inches='tight')
plt.show()

# ==================== 6. 导出结果 ====================
df_strategies.to_csv('asset_strategies.csv')
df_positions.to_csv('asset_positions.csv')
df_portfolio.to_frame('portfolio_return').to_csv('portfolio_strategy.csv')
df_annual_stats.to_csv('annual_statistics.csv')
print("\n✓ 结果已保存")

In [ ]:
df_signal.to_csv('all_signal.csv')

#### Clustermap

In [ ]:
print(price_data.head())
print(price_data['China'].head())
print(monthly_data.head())
print(monthly_data.describe())

In [ ]:
df_positions.describe()
df_portfolio.describe()

In [ ]:
print("df_positions 的列名:", df_positions.columns.tolist())
print("monthly_data 的列名:", monthly_data.columns.tolist())

In [ ]:
# 设置回测起始日期
START_DATE = '2010-01-01'

# 设置列名映射
asset_map = {
    'CH_EQUITY': 'CH_EQUITY',
    'CH_BOND':   'CH_BOND'
}

# 确保索引是 datetime 格式
monthly_data.index = pd.to_datetime(monthly_data.index)
df_positions.index = pd.to_datetime(df_positions.index)

# 【关键步骤】在这里统一截取 2012 年之后的数据
# 使用 copy() 防止警告，并确保后续计算不影响原始数据
sub_returns = monthly_data[monthly_data.index >= START_DATE].copy()
sub_positions = df_positions[df_positions.index >= START_DATE].copy()

# 取两个数据的时间交集，确保对齐
common_index = sub_returns.index.intersection(sub_positions.index)
sub_returns = sub_returns.loc[common_index]
sub_positions = sub_positions.loc[common_index]

print(f"回测时间范围: {common_index.min().date()} 至 {common_index.max().date()}")

# 检查列名是否存在
for key, actual_name in asset_map.items():
    if actual_name not in sub_positions.columns:
        print(f"⚠️ 警告: 在 df_positions 中找不到列名 '{actual_name}'")
    if actual_name not in sub_returns.columns:
        print(f"⚠️ 警告: 在 monthly_data 中找不到列名 '{actual_name}'")

# ==================== 2. 诊断工具函数 ====================
def diagnose_asset_performance(asset_name, signal_series, return_series):
    # 对齐数据并去除空值
    df_diag = pd.DataFrame({
        'Signal': signal_series,
        'Next_Ret': return_series
    }).dropna()
    
    if df_diag.empty:
        print(f"--- {asset_name}: 数据为空，无法诊断 ---")
        return df_diag

    # 计算 IC (相关性)
    ic = df_diag['Signal'].corr(df_diag['Next_Ret'])
    
    # 胜率统计
    # 做多胜率：信号>0 且 收益>0
    long_wins = df_diag[(df_diag['Signal'] > 0) & (df_diag['Next_Ret'] > 0)].shape[0]
    long_total = df_diag[df_diag['Signal'] > 0].shape[0]
    long_win_rate = long_wins / long_total if long_total > 0 else 0
    
    # 做空胜率：信号<0 且 收益<0
    short_wins = df_diag[(df_diag['Signal'] < 0) & (df_diag['Next_Ret'] < 0)].shape[0]
    short_total = df_diag[df_diag['Signal'] < 0].shape[0]
    short_win_rate = short_wins / short_total if short_total > 0 else 0
    
    print(f"\n--- 诊断报告: {asset_name} ({START_DATE} 起) ---")
    print(f"IC (预测相关性): {ic:.4f}")
    print(f"做多胜率: {long_win_rate:.2%} (次数: {long_total})")
    print(f"做空胜率: {short_win_rate:.2%} (次数: {short_total})")
    
    if ic < -0.02:
        print("🔴【严重警告】IC为负！策略在亏钱。建议检查因子方向是否反了。")
    elif abs(ic) < 0.02:
        print("🟡【提示】IC接近0，策略区分度不高。")
    else:
        print("🟢【正常】IC为正，逻辑方向正确。")
    
    return df_diag

# ==================== 3. 运行诊断绘图 ====================
plt.figure(figsize=(12, 10))

plot_idx = 1
for standard_name, real_name in asset_map.items():
    # 跳过不存在的列
    if real_name not in sub_positions.columns or real_name not in sub_returns.columns:
        continue
        
    # 获取截取后的信号和收益
    signal = sub_positions[real_name] 
    returns = sub_returns[real_name]
    
    # 运行诊断
    diagnose_asset_performance(real_name, signal, returns)
    
    # --- 计算净值曲线 ---
    # 1. 买入持有 (Buy & Hold)
    # 从 2012 年开始归一化为 1
    nav_asset = (1 + returns).cumprod()
    nav_asset = nav_asset / nav_asset.iloc[0] 
    
    # 2. 策略收益 (Strategy)
    # 策略收益 = 信号 * 市场收益
    strat_ret = signal * returns
    nav_strat = (1 + strat_ret).cumprod()
    nav_strat = nav_strat / nav_strat.iloc[0] # 归一化
    
    # 绘图
    ax = plt.subplot(len(asset_map), 1, plot_idx)
    plot_idx += 1
    
    ax.plot(nav_asset.index, nav_asset.values, label=f'{real_name} Buy & Hold', color='gray', linestyle='--', alpha=0.7)
    ax.plot(nav_strat.index, nav_strat.values, label=f'{real_name} Strategy', color='red', linewidth=2)
    
    # 绘制 1.0 基准线
    ax.axhline(y=1, color='black', linestyle=':', linewidth=0.5)
    
    ax.set_title(f'{real_name} Performance (Since {START_DATE})')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()